# koilog — журнал силовых тренировок

`koilog` разбирает две выгрузки в одну базу SQLite, считает производные
метрики (1RM, рекорд, скользящее среднее, КОИ) по явным правилам и строит
программу роста нагрузки от последней реальной тренировки — только по тем
весам, которые есть в инвентаре зала.

Вся логика живёт в пакете `koilog` и вызывается из командной строки; этот
ноутбук — обзор поверх той же базы. Он импортирует **те же** функции, что и
CLI, поэтому расхождений между выводом здесь и в терминале нет.

```
python -m koilog load                       # разобрать CSV в sql/training.db
python -m koilog stats                      # сводка по упражнениям за 3 месяца
python -m koilog stats --all                # вся история
python -m koilog plan "Жим гантелей"        # программа + периоды
python -m koilog weeks --exercise "Жим гантелей"   # недельные суммы КОИ
python -m koilog history "Жим гантелей" --from 2026-01-01 --to 2026-10-08
python -m koilog explain "Жим гантелей"     # разбор парсинга и расчёта
python -m koilog check                      # что в данных требует решения
```

Пакет держится на стандартной библиотеке Python 3.13. `matplotlib` и
`ipywidgets` нужны только этому ноутбуку и на CLI не влияют.

## Как читать числа

Короткий словарь, чтобы колонки не приходилось угадывать.

**1RM** — оценка максимального веса на одно повторение по формуле Brzycki
`вес × 36 / (37 − повторы)`. Выше 20 повторов не считается: экстраполяция
перестаёт иметь смысл.

**Для упражнений «тело + жилет»** (`Отжимания`) рабочий вес — это **суммарный**
вес: собственное тело плюс добавка. В таблицах ниже, где нужен вес, он уже
суммарный; в `explain` видно, как из «сырых» 20 кг жилета получается 90 кг
(70 тело + 20 жилет).

| колонка | что значит |
|---|---|
| подх | подходов упражнения в выбранном окне |
| лучший | лучший 1RM **внутри окна** |
| рекорд | лучший 1RM за **всю историю** — база для следующей колонки |
| посл. | 1RM последней тренировки в окне |
| Δ к рекорду | `посл. − рекорд`: 0 = на рекорде, минус = отставание |
| МА30 | скользящее среднее 1RM за **30 дней по этому упражнению** (база КОИ) |
| объём | сумма `вес × повторы` по подходам, в килограммо-повторах |

Раньше на месте «Δ к рекорду» стояла дельта к предыдущей тренировке. Она
честно равнялась нулю на двух одинаковых тренировках подряд (и это выглядело
как поломка), поэтому теперь дельта считается к рекорду, а изменение к прошлой
тренировке печатает `koilog plan` — с датой той тренировки.

**КОИ** — «коэффициент относительной интенсивности», мера усталостной нагрузки.
Формула оригинальная: `КОИ = Σ повторы / (100 − интенсивность%)`, где
интенсивность = `вес / база 1RM`. Делить на 100 обязательно: интенсивность —
доля, а формула выведена в процентах. Выше 98 % от базы КОИ не считается:
при интенсивности около 100 % знаменатель уходит в ноль.

**КОИ считается по каждому упражнению отдельно**, а не по группе: у жима и
тяги в одной группе разные базы 1RM, и общая сумма теряла бы масштаб. Поэтому
и недельные суммы, и суммы за тренировку — по упражнениям.

In [ ]:
# Подключение к уже разобранной базе. Если файла нет - сначала
# `python -m koilog load`, иначе дальнейшие ячейки не найдут таблицу sets.
import pathlib
import sys

sys.path.insert(0, str(pathlib.Path.cwd()))

from koilog import etl, stats

conn = etl.connect(etl.DEFAULT_DB)
print(etl.DEFAULT_DB)

In [ ]:
# Что вообще загрузилось: сколько подходов дал каждый источник и за какой срок.
# То же печатает `python -m koilog load` (блок «источник ... подходов»).
for name, block in etl.summary(conn)["by_source"].items():
    print(f"{name:14} {block['sets']:6} подходов  {block['days']:4} дней  "
          f"{block['first_day']} ... {block['last_day']}")

In [ ]:
# Сводка упражнений за окно - то же, что печатает `python -m koilog stats`.
# Окно здесь - три месяца (как у `stats` без ключа --all): упражнения, которыми
# давно не занимались, в неё не попадают.
# Подписи колонок: подх - подходов, лучший/рекорд/посл. - 1RM, Δ к рекорду -
# отставание последней тренировки от рекорда, МА30 - среднее за 30 дней
# по упражнению, объём - вес x повторы.
since = stats.since_for_months(conn, 3)
print(stats.format_report(stats.exercise_report(
    conn, order="best_rm", limit=15, with_rm_only=True, since=since)))
print("лучший 1RM окна, рекорд за всю историю, последний 1RM и отставание "
      "от рекорда; МА30 - база КОИ по упражнению.")

In [ ]:
# Программа роста нагрузки от последней реальной тренировки: стартовые вес и
# повторы берутся из базы, список весов - из инвентаря зала.
# В шапке видно рекорд, изменение к прошлой тренировке (с её датой) и МА30 -
# базу КОИ. Если в инвентаре нет записанного веса, программа стартует
# с ближайшего не тяжелее и пишет об этом строкой «~».
# Переходы на следующий вес программа больше не печатает - они отдельной
# ячейкой ниже.
print(stats.format_plan(stats.progression_from_db(conn, "Жим гантелей")))

In [ ]:
# Переходы на следующий вес: где заканчивается «волна» на текущем снаряде и на
# какой вес программа переходит дальше. Колонка «дельта» - насколько 1RM нового
# веса отличается от текущего (обычно вниз: новый вес берётся с 6 повторов).
# Это отдельная ячейка, а не часть «программы роста»: там те же переходы
# выводились дважды.
plan = stats.progression_from_db(conn, "Жим гантелей")
print(stats.format_transitions(plan.wave))

In [ ]:
# Недельные суммы КОИ - по КАЖДОМУ упражнению отдельно (оригинальная логика
# КОИ). Колонка «КОИ» - сумма подходов недели по упражнению, «режим» - полоса
# по этой сумме. Отдельной колонки «на подход» нет: полоса задана именно для
# суммы, а усреднение её размывало.
# Полосы недельного КОИ: < 2 лёгкая, 2…3 тяжёлая, 3…4 очень тяжёлая,
# больше 4 «Вы сошли с ума??».
print(stats.format_weekly_koi(stats.weekly_koi(conn, limit=10)))

### История упражнения за интервал

Интервал задаётся двумя строками ISO-дат `ГГГГ-ММ-ДД`, **обе границы
включительно**. `None` в любой из них означает «без границы с этой стороны»:
`ИНТЕРВАЛ_ОТ = None` и `ИНТЕРВАЛ_ДО = None` берут всю историю упражнения.

Направление вывода задаёт `ОБРАТНЫЙ_ПОРЯДОК`: `False` — от первой даты к
последней, `True` — от последней к первой (и тогда `limit` оставляет самые
свежие тренировки). Тот же отчёт в терминале:
`python -m koilog history "УПРАЖНЕНИЕ" --from ДАТА --to ДАТА` (флаг `--reverse`
разворачивает порядок).

In [ ]:
import datetime

# Меняются четыре строки ниже. Даты - включительно, в ISO (ГГГГ-ММ-ДД).
# None = границы нет: от None - с первой записи, до None - по последнюю.
# ОБРАТНЫЙ_ПОРЯДОК=True покажет те же тренировки от последней даты к первой.
УПРАЖНЕНИЕ = "Отжимания"
ИНТЕРВАЛ_ОТ = datetime.date(2026, 1, 1)   # пример: с 1 января 2026
ИНТЕРВАЛ_ДО = None                        # None - по последнюю тренировку
ОБРАТНЫЙ_ПОРЯДОК = False                  # False - от старых; True - от новых


def _iso(value):
    # date -> ISO-строка, None -> None.
    return None if value is None else value.isoformat()


rows = stats.history(
    conn, УПРАЖНЕНИЕ,
    since=_iso(ИНТЕРВАЛ_ОТ), until=_iso(ИНТЕРВАЛ_ДО),
    reverse=ОБРАТНЫЙ_ПОРЯДОК,
)
print(stats.format_history(
    rows, УПРАЖНЕНИЕ,
    since=_iso(ИНТЕРВАЛ_ОТ), until=_iso(ИНТЕРВАЛ_ДО),
    reverse=ОБРАТНЫЙ_ПОРЯДОК,
))
# «верхний подход» - лучший подход тренировки; 1RM - его 1RM; Δ 1RM - к прошлой
# тренировке упражнения; КОИ - сумма подходов тренировки; режим - по KOI_BANDS.

In [ ]:
# Разбор одного упражнения: сначала - как читаются выгрузки (кодировка,
# разделитель, приоритет дат, правила), затем - от «сырого» значения в файле
# до КОИ, подход за подходом.
# Это ответ на вопрос «откуда взялось 111.7 у Отжиманий»: в колонках видно
# «сырой 20 -> режим vest_only -> свой вес 70 + добавка 20 -> вес 90 -> 1RM».
# Таблицу формирует общая функция stats.format_explain - тот же вид, что в
# `python -m koilog explain "Отжимания"`.
print("=== как читаются выгрузки ===")
for note in etl.parse_notes():
    print(f"  - {note}")

audit = etl.exercise_audit(conn, УПРАЖНЕНИЕ)
if audit is None:
    print(f"\nупражнение «{УПРАЖНЕНИЕ}» в базе не найдено")
else:
    print(f"\n=== разбор «{audit.exercise}» ===")
    print(f"подходов всего: {audit.total}, с 1RM: {audit.with_rm}; "
          f"на времени: {audit.timed}, пропущено: {audit.skipped}, "
          f"выбросов: {audit.outlier}; период {audit.first_day} … {audit.last_day}")
    print("режимы веса: " + ", ".join(f"{n} ({c})" for n, c in audit.weight_modes))

    print("\n=== последние подходы: от сырого значения до КОИ ===")
    print(stats.format_explain(stats.explain_sets(conn, УПРАЖНЕНИЕ, limit=8)),
          end="")

In [ ]:
# Проблемные данные: что требует ручного решения (нераспознанные упражнения,
# подозрительное время, неправдоподобная добавка, вес тела по годам, выбросы).
# То же печатает `python -m koilog check`.
print(etl.quality_report(conn).format(limit=10), end="")

## График 1RM

`matplotlib` (и, по желанию, `ipywidgets` для интерактивных виджетов) —
единственные нестандартные библиотеки проекта, и нужны они только ноутбуку:
в пакете `koilog` графики остаются ASCII (`koilog weeks`, `koilog plan`).

```
pip install matplotlib ipywidgets
```

Если `ipywidgets` не установлен, ячейка ниже молча нарисует обычный график для
упражнения из первого поля — ноутбук не сломается.

In [ ]:
# Абсолютный 1RM выбранного упражнения за выбранный период.
#
# Как задать диапазон:
#   * поле «упражнение» - выпадающий список всех упражнений с посчитанным 1RM;
#   * поле «с» и «по» - календарные даты начала и конца, обе включительно.
# Ниже тот же диапазон можно задать кодом (СТАРТ/ФИНИШ) - он используется,
# если виджетов нет, и как начальные значения виджетов.
#
# График интерактивный: смена любого поля перестраивает кривую. Это удобнее
# прежнего статичного графика «дельта к началу периода»: видно абсолютный
# уровень 1RM, а не только его прирост.
#
# Куда развивать дальше: ipympl (обновление in-place без перерисовки) и plotly
# (наведение, зум, экспорт HTML) - если понадобится делиться графиком.
import datetime

%matplotlib inline
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

# Упражнения, у которых вообще есть 1RM: по ним строится выпадающий список.
_exercises = stats.exercise_report(conn, with_rm_only=True, order="exercise")
NAMES = [row.exercise for row in _exercises]

# Начальные значения (они же - запасной вариант без ipywidgets).
УПРАЖНЕНИЕ = "Жим гантелей" if "Жим гантелей" in NAMES else (NAMES or ["—"])[0]
СТАРТ = datetime.date(2015, 1, 1)
ФИНИШ = datetime.date.today()


def draw_rm(exercise, start, end):
    # Нарисовать 1RM упражнения за интервал [start, end] включительно.
    since = start.isoformat() if start else None
    until = end.isoformat() if end else None
    rows = stats.history(conn, exercise, since=since, until=until)

    fig, ax = plt.subplots(figsize=(11, 4.6))
    if not rows:
        ax.set_title(f"нет 1RM: «{exercise}» за {since or 'начало'} … "
                     f"{until or 'конец'}")
        plt.show()
        return

    dates = [r.date for r in rows]
    values = [r.rm_session if r.rm_session is not None else float("nan")
              for r in rows]
    ax.plot(dates, values, marker="o", ms=4, lw=1.6, color="#4c72b0")
    ax.set_title(f"1RM: {exercise} ({dates[0]} … {dates[-1]})")
    ax.set_xlabel("дата")
    ax.set_ylabel("1RM, кг")
    last = values[-1]
    if last == last:  # не NaN
        ax.annotate(f"{last:.1f} кг", xy=(dates[-1], last),
                    xytext=(-8, 10), textcoords="offset points",
                    ha="right", fontweight="bold")
    fig.autofmt_xdate(rotation=30)
    plt.show()


try:
    import ipywidgets as widgets
    from IPython.display import display

    picker = widgets.Dropdown(options=NAMES, value=УПРАЖНЕНИЕ,
                              description="упражнение")
    date_from = widgets.DatePicker(value=СТАРТ, description="с")
    date_to = widgets.DatePicker(value=ФИНИШ, description="по")
    out = widgets.interactive_output(
        draw_rm,
        {"exercise": picker, "start": date_from, "end": date_to},
    )
    print("меняйте поля ниже - график перестроится")
    display(widgets.VBox([picker,
                          widgets.HBox([date_from, date_to]), out]))
except ImportError:
    print("ipywidgets не установлен - показываю обычный график; "
          "pip install ipywidgets включит виджеты")

# Статичный кадр за начальный диапазон: он один попадает в сохранённый
# ноутбук (виджет в файл не записывается) и виден на GitHub без запуска.
draw_rm(УПРАЖНЕНИЕ, СТАРТ, ФИНИШ)

## О формулах и открытых вопросах

**КОИ по упражнению.** Формула `Σ повторы / (100 − интенсивность%)` применяется
к каждому упражнению отдельно — и за неделю, и за тренировку. Пороги полос
вынесены в `koilog.core`: за неделю `< 2 / 2…3 / 3…4 / > 4`, за одну тренировку
`< 0.4 / 0.4…1 / 1…2 / > 2`. Раньше недельные суммы складывались по группам,
из-за чего жим и тяга делили одну сумму; теперь это исправлено.

**Масштаб КОИ** честный (доли, а не проценты: тренировка в 1,1–1,3 — это
`Load`, а не `Crazy`). Недельные суммы сравниваются по полосам `core.KOI_BANDS_WEEK`,
которые заданы именно для суммы.

**Вес собственного тела** больше не константа 70: он оценивается по годам из
упражнений с суммарным весом (подтягивания, отжимания на брусьях) — сначала
«чистый пол», иначе мода, затем расхождение между упражнениями года
разрешается вниз, потому что жилет добавляет вес. Переопределить:
`python -m koilog load --bodyweight-year 2021=82`.

**76 подходов отчёта за 2018 год** записаны весом жилета вместо суммарного
(`Подтягивания` 25, 31,7, 37,3 кг при теле 67,4). Правила для «слишком мало»
нет — выдумывать порог значит выкидывать настоящие подходы, поэтому строки
помечены в `koilog check` как неправдоподобная добавка, но остаются в базе.